In [ ]:
!pip -q install transformers accelerate

In [ ]:
import torch
import transformers
import random

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

***Introduce GPT-2***



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded!")

***Function for calculating perplexity***

In [ ]:
def calculating_perplexity(text,model,tokenizer):

  inputs = tokenizer(text,return_tensors="pt")

  input_ids = inputs["input_ids"]

  outputs = model(input_ids)

  logits = outputs.logits

  probs = torch.softmax(logits, dim=2)

  probs = probs[:, :-1, :]
  input_ids = input_ids[:, 1:]

  r = []
  for i in range(input_ids.shape[1]):
    token_id = input_ids[0,i]
    prob = probs[0,i,token_id]
    r = r + [prob]
  perplexity = torch.exp(
    -(1 / (input_ids.shape[1])) *
    torch.sum(torch.log(torch.stack(r)))
)

  return perplexity

***Endow Temperature parameter***

In [ ]:
def generate_with_decaying_temperature(

    model,

    tokenizer,

    prompt,

    max_new_tokens,

    initial_temperature,

    final_temperature,

    decay_steps,

    top_k

):

    inputs = tokenizer(prompt, return_tensors="pt")

    input_ids = inputs["input_ids"]

    for step in range(max_new_tokens):

        output = model(input_ids)

        logits = output.logits[0,-1,]

        if step < decay_steps:

            temperature = initial_temperature - (initial_temperature - final_temperature)*step/(decay_steps-1)

        else:

            temperature = final_temperature

        logits = logits/temperature

        values, indices = torch.topk(logits,top_k,)

        probs = torch.softmax(values,dim=-1)

        choice = torch.multinomial(probs,num_samples=1,)

        next_token = indices[choice]

        input_ids = torch.cat(

            [input_ids,next_token],

            dim=-1

        )

    text = tokenizer.decode(

    input_ids,

    skip_special_tokens=True

    )

    return text






***Generate contents, Calculate Perplexity & Select candidates***

***1. Generate contents & Calculate Perplexity***

In [ ]:
prompt = "The"

inputs = tokenizer(prompt,return_tensors="pt")

results= []
for i in range(100):
  outputs = model.generate(
      **inputs,
      max_new_tokens=30,
      do_sample = True,
      top_k = 40
  )

  text = tokenizer.decode(outputs[0],skip_special_tokens=True)

  ppl = calculating_perplexity(text, model, tokenizer)

  results = results + [(text,ppl)]
  print(f"{i+1}: PPL={ppl.item():.2f} | {text}")

***2. select candidates***

In [ ]:
results.sort(key=lambda x:x[1])
for (text,ppl) in results[0:10]:
  print(f"PPL{ppl.item():.2f}|{text}")